# 1. Purpose and scope

This notebook supports Jira issue **SCRUM-8: Review merged dataset quality**. It reviews the merged Favorita base before cleaning rules are defined in SCRUM-9.

- **Input:** `data/processed/favorita_merged/favorita_merged_base.parquet`
- **Expected grain:** one row per `(date, store_nbr, item_nbr)`
- **Target column:** `unit_sales`
- **Boundary:** identify and report quality issues only; no cleaning decisions are applied here.

The notebook does not transform the dataset, impute missing values, remove or modify negative `unit_sales`, create feature-engineering columns, or perform modelling.

# 2. Imports and configuration

Only the packages needed for bounded Parquet inspection are imported. Repository paths are resolved from the current working directory so the notebook remains rerunnable from within the EDIP checkout.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

from IPython.display import display

def find_repo_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "requirements.txt").is_file() and (candidate / "data").is_dir():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the EDIP repository.")

repo_root = find_repo_root(Path.cwd())
MERGED_PATH = repo_root / "data" / "processed" / "favorita_merged" / "favorita_merged_base.parquet"
GRAIN_COLUMNS = ["date", "store_nbr", "item_nbr"]

if not MERGED_PATH.is_file():
    raise FileNotFoundError(f"Merged Favorita Parquet not found: {MERGED_PATH}")

input_file = pd.DataFrame([{
    "path": MERGED_PATH.relative_to(repo_root).as_posix(),
    "file_size_bytes": MERGED_PATH.stat().st_size,
    "file_size_mb": MERGED_PATH.stat().st_size / (1024 ** 2),
}])
display(input_file)

,path,file_size_bytes,file_size_mb
0,data/processed/favorita_merged/favorita_merged...,723900039,690.364875


# 3. Load merged Favorita Parquet

The file is opened through `pyarrow.parquet.ParquetFile`. Metadata and one bounded row-group sample are read; the full dataset is not loaded into pandas.

In [2]:
parquet_file = pq.ParquetFile(MERGED_PATH)
parquet_schema = parquet_file.schema_arrow
file_size_bytes = MERGED_PATH.stat().st_size

parquet_metadata_summary = pd.DataFrame([{
    "row_count": parquet_file.metadata.num_rows,
    "column_count": parquet_file.metadata.num_columns,
    "row_group_count": parquet_file.metadata.num_row_groups,
    "file_size_bytes": file_size_bytes,
    "file_size_mb": file_size_bytes / (1024 ** 2),
}])
display(parquet_metadata_summary)

print("Parquet schema:")
print(parquet_schema)

first_row_group = parquet_file.read_row_group(0)
bounded_sample = first_row_group.slice(0, 10).to_pandas()
display(bounded_sample.head(10))

,row_count,column_count,row_group_count,file_size_bytes,file_size_mb
0,125497040,21,502,723900039,690.364875


Parquet schema:
id: int64
date: timestamp[us]
store_nbr: int16
item_nbr: int32
unit_sales: double
onpromotion: bool
family: large_string
class: int16
perishable: int8
city: large_string
state: large_string
store_type: large_string
cluster: int8
transactions: int32
dcoilwtico: double
is_holiday: bool
holiday_type: large_string
holiday_locale: large_string
holiday_description: large_string
holiday_transferred: bool
holiday_event_count: int16
-- schema metadata --
pandas: '{"index_columns": [], "column_indexes": [], "columns": [{"name":' + 2569


,id,date,store_nbr,item_nbr,unit_sales,onpromotion,family,class,perishable,city,...,store_type,cluster,transactions,dcoilwtico,is_holiday,holiday_type,holiday_locale,holiday_description,holiday_transferred,holiday_event_count
0,0,2013-01-01,25,103665,7.0,<NA>,BREAD/BAKERY,2712,1,Salinas,...,D,1,770,NaN,True,Holiday,National,Primer dia del ano,False,1
1,1,2013-01-01,25,105574,1.0,<NA>,GROCERY I,1045,0,Salinas,...,D,1,770,NaN,True,Holiday,National,Primer dia del ano,False,1
2,2,2013-01-01,25,105575,2.0,<NA>,GROCERY I,1045,0,Salinas,...,D,1,770,NaN,True,Holiday,National,Primer dia del ano,False,1
3,3,2013-01-01,25,108079,1.0,<NA>,GROCERY I,1030,0,Salinas,...,D,1,770,NaN,True,Holiday,National,Primer dia del ano,False,1
4,4,2013-01-01,25,108701,1.0,<NA>,DELI,2644,1,Salinas,...,D,1,770,NaN,True,Holiday,National,Primer dia del ano,False,1
5,5,2013-01-01,25,108786,3.0,<NA>,CLEANING,3044,0,Salinas,...,D,1,770,NaN,True,Holiday,National,Primer dia del ano,False,1
6,6,2013-01-01,25,108797,1.0,<NA>,GROCERY I,1004,0,Salinas,...,D,1,770,NaN,True,Holiday,National,Primer dia del ano,False,1
7,7,2013-01-01,25,108952,1.0,<NA>,CLEANING,3024,0,Salinas,...,D,1,770,NaN,True,Holiday,National,Primer dia del ano,False,1
8,8,2013-01-01,25,111397,13.0,<NA>,GROCERY I,1072,0,Salinas,...,D,1,770,NaN,True,Holiday,National,Primer dia del ano,False,1
9,9,2013-01-01,25,114790,3.0,<NA>,GROCERY I,1004,0,Salinas,...,D,1,770,NaN,True,Holiday,National,Primer dia del ano,False,1


# 4. Basic dataset structure

Metadata supplies the total dimensions and ordered schema. A bounded scan reads only `id` plus the three grain columns from one row group at a time to calculate store/item cardinality, date bounds, ID uniqueness, and grain duplication without materializing the full 21-column dataset in pandas.

### Dimensions, ordered columns, and schema

In [3]:
EXPECTED_COLUMNS = [
    "id",
    "date",
    "store_nbr",
    "item_nbr",
    "unit_sales",
    "onpromotion",
    "family",
    "class",
    "perishable",
    "city",
    "state",
    "store_type",
    "cluster",
    "transactions",
    "dcoilwtico",
    "is_holiday",
    "holiday_type",
    "holiday_locale",
    "holiday_description",
    "holiday_transferred",
    "holiday_event_count",
]

ordered_columns = parquet_schema.names
structure_dimensions = pd.DataFrame([{
    "total_row_count": parquet_file.metadata.num_rows,
    "total_column_count": parquet_file.metadata.num_columns,
    "all_expected_columns_present": all(column in ordered_columns for column in EXPECTED_COLUMNS),
    "expected_column_order_matches": ordered_columns == EXPECTED_COLUMNS,
}])
display(structure_dimensions)

print("Full ordered column list:")
print(ordered_columns)

schema_table = pd.DataFrame([
    {"position": position, "column": field.name, "parquet_dtype": str(field.type)}
    for position, field in enumerate(parquet_schema)
])
display(schema_table)

missing_expected_columns = [column for column in EXPECTED_COLUMNS if column not in ordered_columns]
if missing_expected_columns:
    raise ValueError(f"Missing expected columns: {missing_expected_columns}")

,total_row_count,total_column_count,all_expected_columns_present,expected_column_order_matches
0,125497040,21,True,True


Full ordered column list:
['id', 'date', 'store_nbr', 'item_nbr', 'unit_sales', 'onpromotion', 'family', 'class', 'perishable', 'city', 'state', 'store_type', 'cluster', 'transactions', 'dcoilwtico', 'is_holiday', 'holiday_type', 'holiday_locale', 'holiday_description', 'holiday_transferred', 'holiday_event_count']


,position,column,parquet_dtype
0,0,id,int64
1,1,date,timestamp[us]
2,2,store_nbr,int16
3,3,item_nbr,int32
4,4,unit_sales,double
5,5,onpromotion,bool
6,6,family,large_string
7,7,class,int16
8,8,perishable,int8
9,9,city,large_string


### Bounded first-ten-row preview

This reuses the ten rows already read from the first row group; it does not trigger a full Parquet scan.

In [4]:
display(bounded_sample.head(10))

,id,date,store_nbr,item_nbr,unit_sales,onpromotion,family,class,perishable,city,...,store_type,cluster,transactions,dcoilwtico,is_holiday,holiday_type,holiday_locale,holiday_description,holiday_transferred,holiday_event_count
0,0,2013-01-01,25,103665,7.0,<NA>,BREAD/BAKERY,2712,1,Salinas,...,D,1,770,NaN,True,Holiday,National,Primer dia del ano,False,1
1,1,2013-01-01,25,105574,1.0,<NA>,GROCERY I,1045,0,Salinas,...,D,1,770,NaN,True,Holiday,National,Primer dia del ano,False,1
2,2,2013-01-01,25,105575,2.0,<NA>,GROCERY I,1045,0,Salinas,...,D,1,770,NaN,True,Holiday,National,Primer dia del ano,False,1
3,3,2013-01-01,25,108079,1.0,<NA>,GROCERY I,1030,0,Salinas,...,D,1,770,NaN,True,Holiday,National,Primer dia del ano,False,1
4,4,2013-01-01,25,108701,1.0,<NA>,DELI,2644,1,Salinas,...,D,1,770,NaN,True,Holiday,National,Primer dia del ano,False,1
5,5,2013-01-01,25,108786,3.0,<NA>,CLEANING,3044,0,Salinas,...,D,1,770,NaN,True,Holiday,National,Primer dia del ano,False,1
6,6,2013-01-01,25,108797,1.0,<NA>,GROCERY I,1004,0,Salinas,...,D,1,770,NaN,True,Holiday,National,Primer dia del ano,False,1
7,7,2013-01-01,25,108952,1.0,<NA>,CLEANING,3024,0,Salinas,...,D,1,770,NaN,True,Holiday,National,Primer dia del ano,False,1
8,8,2013-01-01,25,111397,13.0,<NA>,GROCERY I,1072,0,Salinas,...,D,1,770,NaN,True,Holiday,National,Primer dia del ano,False,1
9,9,2013-01-01,25,114790,3.0,<NA>,GROCERY I,1004,0,Salinas,...,D,1,770,NaN,True,Holiday,National,Primer dia del ano,False,1


### Memory-safe cardinality, date, and duplicate scan

Only `id`, `date`, `store_nbr`, and `item_nbr` are read. Exact-row duplication can be certified as zero when `id` is globally unique because an exact duplicate would necessarily repeat `id`. Grain duplicates are counted by adjacent key comparison after validating that the grain is globally ordered; row-group boundary pairs are included. If either prerequisite fails, the corresponding full exact count is explicitly deferred rather than estimated.

In [5]:
SCAN_COLUMNS = ["id", *GRAIN_COLUMNS]
unique_stores = set()
unique_items = set()
minimum_date = None
maximum_date = None
scanned_rows = 0
id_strictly_increasing = True
grain_globally_ordered = True
adjacent_grain_duplicate_count = 0
previous_last_id = None
previous_last_grain = None

for row_group_index in range(parquet_file.metadata.num_row_groups):
    scan_frame = parquet_file.read_row_group(
        row_group_index,
        columns=SCAN_COLUMNS,
    ).to_pandas()
    if scan_frame.empty:
        continue

    scanned_rows += len(scan_frame)
    unique_stores.update(int(value) for value in scan_frame["store_nbr"].unique())
    unique_items.update(int(value) for value in scan_frame["item_nbr"].unique())
    group_minimum_date = scan_frame["date"].min()
    group_maximum_date = scan_frame["date"].max()
    minimum_date = group_minimum_date if minimum_date is None else min(minimum_date, group_minimum_date)
    maximum_date = group_maximum_date if maximum_date is None else max(maximum_date, group_maximum_date)

    id_values = scan_frame["id"].to_numpy(copy=False)
    if len(id_values) > 1 and not bool(np.all(id_values[1:] > id_values[:-1])):
        id_strictly_increasing = False
    if previous_last_id is not None and int(id_values[0]) <= previous_last_id:
        id_strictly_increasing = False
    previous_last_id = int(id_values[-1])

    dates = scan_frame["date"].to_numpy(copy=False)
    stores = scan_frame["store_nbr"].to_numpy(copy=False)
    items = scan_frame["item_nbr"].to_numpy(copy=False)
    if len(scan_frame) > 1:
        same_date = dates[1:] == dates[:-1]
        same_store = stores[1:] == stores[:-1]
        adjacent_grain_duplicate_count += int(np.sum(same_date & same_store & (items[1:] == items[:-1])))
        order_violation = (
            (dates[1:] < dates[:-1])
            | (same_date & (stores[1:] < stores[:-1]))
            | (same_date & same_store & (items[1:] < items[:-1]))
        )
        if bool(np.any(order_violation)):
            grain_globally_ordered = False

    first_grain = (pd.Timestamp(dates[0]), int(stores[0]), int(items[0]))
    last_grain = (pd.Timestamp(dates[-1]), int(stores[-1]), int(items[-1]))
    if previous_last_grain is not None:
        if first_grain < previous_last_grain:
            grain_globally_ordered = False
        if first_grain == previous_last_grain:
            adjacent_grain_duplicate_count += 1
    previous_last_grain = last_grain

if scanned_rows != parquet_file.metadata.num_rows:
    raise ValueError("Narrow structural scan row count does not match Parquet metadata.")

exact_duplicate_row_count = 0 if id_strictly_increasing else pd.NA
grain_duplicate_row_count = adjacent_grain_duplicate_count if grain_globally_ordered else pd.NA

basic_structure_results = pd.DataFrame([{
    "total_row_count": parquet_file.metadata.num_rows,
    "total_column_count": parquet_file.metadata.num_columns,
    "unique_store_count": len(unique_stores),
    "unique_item_count": len(unique_items),
    "minimum_date": minimum_date.date().isoformat(),
    "maximum_date": maximum_date.date().isoformat(),
    "id_strictly_increasing": id_strictly_increasing,
    "exact_duplicate_row_count": exact_duplicate_row_count,
    "grain_globally_ordered": grain_globally_ordered,
    "duplicate_date_store_item_count": grain_duplicate_row_count,
}])
display(basic_structure_results)

if pd.isna(exact_duplicate_row_count):
    print("Full exact-row duplicate counting is deferred: global ID uniqueness was not certified by the narrow scan.")
if pd.isna(grain_duplicate_row_count):
    print("Full grain duplicate counting is deferred: the dataset is not globally ordered by the expected grain.")

,total_row_count,total_column_count,unique_store_count,unique_item_count,minimum_date,maximum_date,id_strictly_increasing,exact_duplicate_row_count,grain_globally_ordered,duplicate_date_store_item_count
0,125497040,21,54,4036,2013-01-01,2017-08-15,True,0,True,0


### Section 4 checkpoint

- **Parquet opened successfully:** Yes. PyArrow read the metadata, schema, first row group, and bounded preview without error.
- **Row and column counts look valid:** Yes. The file contains **125,497,040 rows**, **21 columns**, and all expected columns in the expected order.
- **Expected grain appears preserved:** Yes. The memory-safe scan certified global grain ordering and found **0 duplicate `(date, store_nbr, item_nbr)` rows**. Strictly increasing IDs also certify **0 exact duplicate rows**.
- **Ready for deeper SCRUM-8 quality checks:** Yes. The structural baseline is valid for the next quality-review sections; no cleaning rules have been applied.
